In [1]:
import csv
from random import shuffle
from functools import reduce
from copy import deepcopy

In [2]:
FIELDNAMES = [
    'nct_id', 'phase', 'diseases', 'drugs',
    'smiles', 'description', 'criteria',
    'label', 'icdcodes'
]

def csvfile2rows(input_file):
    with open(input_file, 'r', encoding='utf-8') as csvfile:
        reader = csv.DictReader(csvfile)
        return list(reader)

def filter_phase_I(row):
    return "phase1" in row["phase"].lower()

def filter_phase_II(row):
    return "phase2" in row["phase"].lower()

def filter_phase_III(row):
    return "phase3" in row["phase"].lower()

def filter_trial(row):
    label = int(row["label"])
    phase = row["phase"].lower()
    if label == 0 and ('phase1' in phase or 'phase2' in phase):
        return True
    if ('phase3' in phase or 'phase4' in phase) and label == 1:
        return True
    return False


In [3]:
def write_row_to_csvfile(rows, fieldnames, output_file):
    with open(output_file, 'w', encoding='utf-8', newline='') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow({k: row[k] for k in fieldnames})

In [4]:
def split_data(rows, train_ratio=0.8):
    shuffle(rows)
    n = len(rows)
    train_end = int(n * train_ratio)
    return rows[:train_end], rows[train_end:]

In [5]:
def check_pos_and_neg(rows):
    pos_cnt = sum(1 for row in rows if int(row["label"]) == 1)
    neg_cnt = sum(1 for row in rows if int(row["label"]) == 0)
    print("pos:", pos_cnt, "neg:", neg_cnt)

In [6]:
def smiles_txt_to_lst(text):
    text = text[1:-1]
    lst = [i.strip()[1:-1] for i in text.split(',')]
    return lst

In [7]:
def clean_data(input_file, clean_file):
    rows = csvfile2rows(input_file)
    cleaned_rows = []

    for row in rows:
        smiless = row['smiles']
        if '[O--].[Mg++]' in smiless:
            smiles_lst = set(smiles_txt_to_lst(smiless))
            if '[O--].[Mg++]' in smiles_lst:
                smiles_lst.remove('[O--].[Mg++]')
            if not smiles_lst:
                continue
            row['smiles'] = str(list(smiles_lst))
        cleaned_rows.append(row)

    write_row_to_csvfile(cleaned_rows, FIELDNAMES, clean_file)

In [8]:
def select_and_split_data(input_file, filter_func, output_file_name):
    rows = csvfile2rows(input_file)
    rows = list(filter(filter_func, rows))

    positive_num = sum(1 for row in rows if int(row["label"]) == 1)
    negative_num = len(rows) - positive_num
    print(f"\t\tpos = {positive_num}  neg = {negative_num}")

    train_row, valid_row = split_data(rows)

    print("train")
    check_pos_and_neg(train_row)
    print("valid")
    check_pos_and_neg(valid_row)

    write_row_to_csvfile(train_row, FIELDNAMES, output_file_name.replace('.csv', '_train.csv'))
    write_row_to_csvfile(valid_row, FIELDNAMES, output_file_name.replace('.csv', '_valid.csv'))

In [9]:
if __name__ == "__main__":
    input_file = 'CTOD_clean_dataset.csv'
    clean_file = "data/labelling/clean_data.csv"

    clean_data(input_file, clean_file)

    print("------------ phase I -------------")
    select_and_split_data(clean_file, filter_phase_I, 'data/labelling/phase_I.csv')

    print("----------- phase II -------------")
    select_and_split_data(clean_file, filter_phase_II, 'data/labelling/phase_II.csv')

    print("----------- phase III ----------")
    select_and_split_data(clean_file, filter_phase_III, 'data/labelling/phase_III.csv')

    print("----------- indication ----------")
    select_and_split_data(clean_file, filter_trial, 'data/labelling/indication.csv')

------------ phase I -------------
		pos = 16694  neg = 3401
train
pos: 13384 neg: 2692
valid
pos: 3310 neg: 709
----------- phase II -------------
		pos = 20546  neg = 6414
train
pos: 16438 neg: 5130
valid
pos: 4108 neg: 1284
----------- phase III ----------
		pos = 15018  neg = 2834
train
pos: 12000 neg: 2281
valid
pos: 3018 neg: 553
----------- indication ----------
		pos = 15018  neg = 8667
train
pos: 11967 neg: 6981
valid
pos: 3051 neg: 1686


In [10]:
import csv
from random import shuffle
from functools import reduce
from copy import deepcopy

# --- Campos esperados --- #
FIELDNAMES = [
    'nct_id', 'phase', 'diseases', 'drugs',
    'smiles', 'description', 'criteria',
    'label', 'icdcodes'
]

# --- Ler CSV como lista de dicionários --- #
def csvfile2rows(input_file):
    with open(input_file, 'r', encoding='utf-8') as csvfile:
        reader = csv.DictReader(csvfile)
        return list(reader)

# --- Filtros por fase --- #
def filter_phase_I(row):
    return "phase1" in row["phase"].lower()

def filter_phase_II(row):
    return "phase2" in row["phase"].lower()

def filter_phase_III(row):
    return "phase3" in row["phase"].lower()

# --- Filtro para dados de indicação (condicional por fase e label) --- #
def filter_trial(row):
    label = int(row["label"])
    phase = row["phase"].lower()
    if label == 0 and ('phase1' in phase or 'phase2' in phase):
        return True
    if ('phase3' in phase or 'phase4' in phase) and label == 1:
        return True
    return False

# --- Escrita de dados em arquivo CSV --- #
def write_row_to_csvfile(rows, fieldnames, output_file):
    with open(output_file, 'w', encoding='utf-8', newline='') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow({k: row[k] for k in fieldnames})

# --- Divisão em treino/validação --- #
def split_data(rows, train_ratio=0.8):
    shuffle(rows)
    n = len(rows)
    train_end = int(n * train_ratio)
    return rows[:train_end], rows[train_end:]

# --- Contagem de positivos/negativos --- #
def check_pos_and_neg(rows):
    pos_cnt = sum(1 for row in rows if int(row["label"]) == 1)
    neg_cnt = sum(1 for row in rows if int(row["label"]) == 0)
    print("pos:", pos_cnt, "neg:", neg_cnt)

# --- Conversão de lista de SMILES em string para lista Python --- #
def smiles_txt_to_lst(text):
    text = text[1:-1]
    lst = [i.strip()[1:-1] for i in text.split(',')]
    return lst

# --- Limpeza dos dados --- #
def clean_data(input_file, clean_file):
    rows = csvfile2rows(input_file)
    cleaned_rows = []
    removed = 0

    for row in rows:
        smiless = row['smiles']
        if '[O--].[Mg++]' in smiless:
            smiles_lst = set(smiles_txt_to_lst(smiless))
            if '[O--].[Mg++]' in smiles_lst:
                smiles_lst.remove('[O--].[Mg++]')
            if not smiles_lst:
                removed += 1
                continue
            row['smiles'] = str(list(smiles_lst))
        cleaned_rows.append(row)

    print(f"Removed {removed} rows with only [O--].[Mg++]")
    write_row_to_csvfile(cleaned_rows, FIELDNAMES, clean_file)

# --- Selecionar, filtrar e dividir dados por fase ou critério --- #
def select_and_split_data(input_file, filter_func, output_file_name):
    rows = csvfile2rows(input_file)
    rows = list(filter(filter_func, rows))

    positive_num = sum(1 for row in rows if int(row["label"]) == 1)
    negative_num = len(rows) - positive_num
    print(f"\t\tpos = {positive_num}  neg = {negative_num}")

    train_row, valid_row = split_data(rows)

    print("train")
    check_pos_and_neg(train_row)
    print("valid")
    check_pos_and_neg(valid_row)

    write_row_to_csvfile(train_row, FIELDNAMES, output_file_name.replace('.csv', '_train.csv'))
    write_row_to_csvfile(valid_row, FIELDNAMES, output_file_name.replace('.csv', '_valid.csv'))

# --- Execução principal --- #
if __name__ == "__main__":
    input_file = 'CTOD_clean_dataset.csv'
    clean_file = "data/labelling/clean_data.csv"

    clean_data(input_file, clean_file)

    print("------------ phase I -------------")
    select_and_split_data(clean_file, filter_phase_I, 'data/labelling/phase_I.csv')

    print("----------- phase II -------------")
    select_and_split_data(clean_file, filter_phase_II, 'data/labelling/phase_II.csv')

    print("----------- phase III ----------")
    select_and_split_data(clean_file, filter_phase_III, 'data/labelling/phase_III.csv')

    print("----------- indication ----------")
    select_and_split_data(clean_file, filter_trial, 'data/labelling/indication.csv')


Removed 0 rows with only [O--].[Mg++]
------------ phase I -------------
		pos = 16694  neg = 3401
train
pos: 13370 neg: 2706
valid
pos: 3324 neg: 695
----------- phase II -------------
		pos = 20546  neg = 6414
train
pos: 16442 neg: 5126
valid
pos: 4104 neg: 1288
----------- phase III ----------
		pos = 15018  neg = 2834
train
pos: 12056 neg: 2225
valid
pos: 2962 neg: 609
----------- indication ----------
		pos = 15018  neg = 8667
train
pos: 12079 neg: 6869
valid
pos: 2939 neg: 1798


In [11]:
import csv
from random import shuffle

FIELDNAMES = [
    'nct_id', 'phase', 'diseases', 'drugs',
    'smiles', 'description', 'criteria',
    'label', 'icdcodes'
]

def csvfile2rows(input_file):
    with open(input_file, 'r', encoding='utf-8') as csvfile:
        reader = csv.DictReader(csvfile)
        return list(reader)

def write_row_to_csvfile(rows, fieldnames, output_file):
    with open(output_file, 'w', encoding='utf-8', newline='') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow({k: row[k] for k in fieldnames})

def filter_phase_I(row):
    return "phase1" in row["phase"].lower()

def filter_phase_II(row):
    return "phase2" in row["phase"].lower()

def filter_phase_III(row):
    return "phase3" in row["phase"].lower()

def filter_trial(row):
    label = int(row["label"])
    phase = row["phase"].lower()
    if label == 0 and ('phase1' in phase or 'phase2' in phase):
        return True
    if ('phase3' in phase or 'phase4' in phase) and label == 1:
        return True
    return False

def split_data(rows, train_ratio=0.8):
    shuffle(rows)
    n = len(rows)
    train_end = int(n * train_ratio)
    return rows[:train_end], rows[train_end:]

def check_pos_and_neg(rows):
    pos_cnt = sum(1 for row in rows if int(row["label"]) == 1)
    neg_cnt = sum(1 for row in rows if int(row["label"]) == 0)
    print("pos:", pos_cnt, "neg:", neg_cnt)

def clean_data(input_file, clean_file):
    # Apenas copia os dados sem nenhuma modificação
    rows = csvfile2rows(input_file)
    print(f"Copied {len(rows)} rows from original file.")
    write_row_to_csvfile(rows, FIELDNAMES, clean_file)

def select_and_split_data(input_file, filter_func, output_file_name):
    rows = csvfile2rows(input_file)
    rows = list(filter(filter_func, rows))

    positive_num = sum(1 for row in rows if int(row["label"]) == 1)
    negative_num = len(rows) - positive_num
    print(f"\t\tpos = {positive_num}  neg = {negative_num}")

    train_row, valid_row = split_data(rows)

    print("train")
    check_pos_and_neg(train_row)
    print("valid")
    check_pos_and_neg(valid_row)

    write_row_to_csvfile(train_row, FIELDNAMES, output_file_name.replace('.csv', '_train.csv'))
    write_row_to_csvfile(valid_row, FIELDNAMES, output_file_name.replace('.csv', '_valid.csv'))

if __name__ == "__main__":
    input_file = 'CTOD_clean_dataset.csv'
    clean_file = "data/labelling/clean_data.csv"

    clean_data(input_file, clean_file)

    print("------------ phase I -------------")
    select_and_split_data(clean_file, filter_phase_I, 'data/labelling/phase_I.csv')

    print("----------- phase II -------------")
    select_and_split_data(clean_file, filter_phase_II, 'data/labelling/phase_II.csv')

    print("----------- phase III ----------")
    select_and_split_data(clean_file, filter_phase_III, 'data/labelling/phase_III.csv')

    print("----------- indication ----------")
    select_and_split_data(clean_file, filter_trial, 'data/labelling/indication.csv')


Copied 58860 rows from original file.
------------ phase I -------------
		pos = 16694  neg = 3401
train
pos: 13388 neg: 2688
valid
pos: 3306 neg: 713
----------- phase II -------------
		pos = 20546  neg = 6414
train
pos: 16505 neg: 5063
valid
pos: 4041 neg: 1351
----------- phase III ----------
		pos = 15018  neg = 2834
train
pos: 12010 neg: 2271
valid
pos: 3008 neg: 563
----------- indication ----------
		pos = 15018  neg = 8667
train
pos: 11990 neg: 6958
valid
pos: 3028 neg: 1709
